In [3]:
import yaml
import os
import matplotlib.pyplot as plt 
import seaborn as sns
import os
import yaml
import os.path as op
import scanpy as sc
import numpy as np
import pandas as pd




In [4]:


def read_all_results_grnboost(root_dir, select = ('overlaps_global_top_k_quantile_count.tsv')):
    """
    Reads all YAML files in a tree of directories starting from root_dir.extended
    Args: 
        root_dir (str): The path to the root directory.

    Returns:
        list: A list of dictionaries, where each dictionary represents the
              content of a YAML file.
    """
    all_results = []
    
    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return all_results

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(select) :

                filepath = os.path.join(dirpath, filename)
                try:
                    overlaps = pd.read_csv(filepath, sep = '\t', index_col=0)
                    overlaps['dataset'] = op.basename(op.dirname(filepath))

                    all_results.append(overlaps)
                except:
                    continue
    all_results = pd.concat(all_results)

                    
    return all_results


def read_all_results_sweep(root_dir, select = ('overlaps_global_top_k.tsv')):
    """
    Reads all YAML files in a tree of directories starting from root_dir.extended
    Args: 
        root_dir (str): The path to the root directory.

    Returns:
        list: A list of dictionaries, where each dictionary represents the
              content of a YAML file.
    """
    all_results = []
    
    # Normalize the root_dir path
    root_dir = os.path.abspath(root_dir)

    # Check if the root directory exists
    if not os.path.isdir(root_dir):
        print(f"Error: Directory '{root_dir}' not found.")
        return all_results

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(select) :

                filepath = os.path.join(dirpath, filename)
                try:
                    overlaps = pd.read_csv(filepath, sep = '\t', index_col=0)
                    overlaps['dataset'] = op.basename(op.dirname(filepath))
                    overlaps['method']  = op.basename(filepath).replace(select, '')
                    all_results.append(overlaps)
                except:
                    continue
    all_results = pd.concat(all_results)

                    
    return all_results


In [5]:
import os
import os.path as op

In [8]:

outdir ='/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/best_models_log'
os.makedirs(outdir, exist_ok=True)

final_metrics = read_all_results_sweep('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_best_models_log3/',   select = ('_aggregated_performance_grn.tsv'))
split_data = final_metrics['method'].str.split('_', expand=True)
split_data.columns = ['xai_method', 'hidden_layer_size', 'n_hidden_layers', 'dropout', 'model', 'background', 'raw']
final_metrics = pd.concat([final_metrics, split_data], axis=1)
final_metrics  = final_metrics.drop(columns= ['n_top'])
final_metrics = final_metrics.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap',  'type': 'target'})
final_metrics.to_csv(op.join(outdir, 'edge_recovery_best_netmap_grn.tsv'), sep = '\t', index = False)

In [4]:
final_metrics_sc = read_all_results_sweep('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries_scgenerai/',   select = ('_aggregated_performance.tsv'))


In [5]:
final_metrics_sc

,n_top,percentage_recovered,tp,gs_count,pp,type,grn,net,top_perc,dataset,method
0,34,0.000000,0,154,34,on_target,net_84_10865,0,0.001,net_84_10865_net_88_10937_net_90_11013,config_easy_net_84_10865_net_88_10937_net_90_1...
1,342,0.006494,1,154,342,on_target,net_84_10865,0,0.010,net_84_10865_net_88_10937_net_90_11013,config_easy_net_84_10865_net_88_10937_net_90_1...
2,1710,0.012987,2,154,1710,on_target,net_84_10865,0,0.050,net_84_10865_net_88_10937_net_90_11013,config_easy_net_84_10865_net_88_10937_net_90_1...
3,3419,0.045455,7,154,3419,on_target,net_84_10865,0,0.100,net_84_10865_net_88_10937_net_90_11013,config_easy_net_84_10865_net_88_10937_net_90_1...
4,6838,0.110390,17,154,6838,on_target,net_84_10865,0,0.200,net_84_10865_net_88_10937_net_90_11013,config_easy_net_84_10865_net_88_10937_net_90_1...
...,...,...,...,...,...,...,...,...,...,...,...
4,24453,0.109756,18,164,24453,off_target,net_82_10152,1,0.200,net_60_10082_net_64_11307_net_84_11226_net_133...,config_five_net_60_10082_net_64_11307_net_84_1...
5,30566,0.170732,28,164,30566,off_target,net_82_10152,1,0.250,net_60_10082_net_64_11307_net_84_11226_net_133...,config_five_net_60_10082_net_64_11307_net_84_1...
6,61132,0.341463,56,164,61132,off_target,net_82_10152,1,0.500,net_60_10082_net_64_11307_net_84_11226_net_133...,config_five_net_60_10082_net_64_11307_net_84_1...
7,91699,0.487805,80,164,91699,off_target,net_82_10152,1,0.750,net_60_10082_net_64_11307_net_84_11226_net_133...,config_five_net_60_10082_net_64_11307_net_84_1...


In [6]:

final_metrics_sc  = final_metrics_sc.drop(columns= ['n_top'])
final_metrics_sc['method'] = 'scgenerai_config'
final_metrics_sc = final_metrics_sc.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap',  'type': 'target'})
final_metrics_sc.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery_scgenerai.tsv', sep = '\t', index = False)

In [7]:
grnb = read_all_results_grnboost('/data_nfs/og86asub/netmap/netmap-evaluation/results/summaries/',   select = 'preclustered_overlaps_global_top_k.tsv')
grnb['top_perc'] = np.tile([0.001,0.01, 0.05, 0.1, 0.2, 0.25, 0.5, 0.75, 1.0], reps = int(grnb.shape[0]/9))
grnb  = grnb.drop(columns= ['n_top'])
grnb = grnb.rename(columns= {'top_perc': "n_top", 'percentage_recovered': 'percentage_overlap', 'config': 'method', 'type': 'target'})
grnb.to_csv('/data_nfs/og86asub/netmap/netmap-evaluation/results/figures/data/edge_recovery_grnboost.tsv', sep = '\t', index=False)


In [8]:
final_metrics

,percentage_overlap,tp,gs_count,pp,target,grn,net,n_top,dataset,method,xai_method,hidden_layer_size,n_hidden_layers,dropout,model,background,raw
0,0.000000,0,154,69,on_target,net_84_10865,0,0.001,net_84_10865_net_88_10937_net_90_11013,GuidedBackprop_64_1_0.1_NegativeBinomialAutoen...,GuidedBackprop,64,1,0.1,NegativeBinomialAutoencoder,zeros,False
1,0.019481,3,154,686,on_target,net_84_10865,0,0.010,net_84_10865_net_88_10937_net_90_11013,GuidedBackprop_64_1_0.1_NegativeBinomialAutoen...,GuidedBackprop,64,1,0.1,NegativeBinomialAutoencoder,zeros,False
2,0.162338,25,154,3432,on_target,net_84_10865,0,0.050,net_84_10865_net_88_10937_net_90_11013,GuidedBackprop_64_1_0.1_NegativeBinomialAutoen...,GuidedBackprop,64,1,0.1,NegativeBinomialAutoencoder,zeros,False
3,0.240260,37,154,6864,on_target,net_84_10865,0,0.100,net_84_10865_net_88_10937_net_90_11013,GuidedBackprop_64_1_0.1_NegativeBinomialAutoen...,GuidedBackprop,64,1,0.1,NegativeBinomialAutoencoder,zeros,False
4,0.389610,60,154,13729,on_target,net_84_10865,0,0.200,net_84_10865_net_88_10937_net_90_11013,GuidedBackprop_64_1_0.1_NegativeBinomialAutoen...,GuidedBackprop,64,1,0.1,NegativeBinomialAutoencoder,zeros,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4,0.497143,87,175,40680,off_target,net_75_10306,1,0.200,net_98_11932_net_51_10906_net_60_10082_net_90_...,GradientShap_64_1_0.1_ZINBAutoencoder_zeros_True,GradientShap,64,1,0.1,ZINBAutoencoder,zeros,True
5,0.560000,98,175,50850,off_target,net_75_10306,1,0.250,net_98_11932_net_51_10906_net_60_10082_net_90_...,GradientShap_64_1_0.1_ZINBAutoencoder_zeros_True,GradientShap,64,1,0.1,ZINBAutoencoder,zeros,True
6,0.874286,153,175,101700,off_target,net_75_10306,1,0.500,net_98_11932_net_51_10906_net_60_10082_net_90_...,GradientShap_64_1_0.1_ZINBAutoencoder_zeros_True,GradientShap,64,1,0.1,ZINBAutoencoder,zeros,True
7,0.965714,169,175,152551,off_target,net_75_10306,1,0.750,net_98_11932_net_51_10906_net_60_10082_net_90_...,GradientShap_64_1_0.1_ZINBAutoencoder_zeros_True,GradientShap,64,1,0.1,ZINBAutoencoder,zeros,True


In [9]:
grnb

,percentage_overlap,tp,gs_count,pp,target,grn,net,method,dataset,n_top
0,0.012821,1,78,35,on_target,0,0,grnboost2_config,net_53_11196_net_70_11431_net_84_9903,0.001
1,0.089744,7,78,352,on_target,0,0,grnboost2_config,net_53_11196_net_70_11431_net_84_9903,0.010
2,0.192308,15,78,1761,on_target,0,0,grnboost2_config,net_53_11196_net_70_11431_net_84_9903,0.050
3,0.217949,17,78,3522,on_target,0,0,grnboost2_config,net_53_11196_net_70_11431_net_84_9903,0.100
4,0.320513,25,78,7043,on_target,0,0,grnboost2_config,net_53_11196_net_70_11431_net_84_9903,0.200
...,...,...,...,...,...,...,...,...,...,...
4,0.162393,19,117,131732,on_target,9,9,grnboost2_config,net_70_11670_net_89_11634_net_57_10409_net_177...,0.200
5,0.162393,19,117,131732,on_target,9,9,grnboost2_config,net_70_11670_net_89_11634_net_57_10409_net_177...,0.250
6,0.162393,19,117,131732,on_target,9,9,grnboost2_config,net_70_11670_net_89_11634_net_57_10409_net_177...,0.500
7,0.162393,19,117,131732,on_target,9,9,grnboost2_config,net_70_11670_net_89_11634_net_57_10409_net_177...,0.750
